# PyTorch 工業影像異常偵測 Demo

這份 Notebook 會在 Google Colab 依序執行三種 CPU Demo：卷積自編碼器、逐像素參考法、局部區塊 KNN，並顯示合成正常／瑕疵範例的結果。

> 合成資料只驗證程式流程，不代表真實產品檢測效能。正式使用前須以實際資料驗證與校準。

## 準備專案 ZIP
在 VS Code 將專案壓縮成 ZIP 後，在下一格上傳。請不要把 `.venv` 或大型 `artifacts` 訓練輸出放入 ZIP；Notebook 會自行產生 Demo 結果。

In [ ]:
%pip install -q "numpy>=1.24,<2" "Pillow>=10,<14"
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("請一次上傳一個包含三個 detector .py 檔案的專案 ZIP。")

extract_root = Path("/content/pytorch_project")
extract_root.mkdir(parents=True, exist_ok=True)
with ZipFile(zip_names[0]) as archive:
    for member in archive.infolist():
        destination = (extract_root / member.filename).resolve()
        if extract_root.resolve() not in destination.parents and destination != extract_root.resolve():
            raise ValueError("ZIP 含有不安全的路徑，已停止解壓。")
    archive.extractall(extract_root)

matches = list(extract_root.rglob("anomaly_detector.py"))
if len(matches) != 1:
    raise FileNotFoundError("ZIP 中必須有且只有一份 anomaly_detector.py。")
project_root = matches[0].parent
for filename in ("pixel_reference_detector.py", "patch_knn_detector.py"):
    if not (project_root / filename).is_file():
        raise FileNotFoundError(f"ZIP 中找不到 {filename}。")
print(f"專案已載入：{project_root}")

In [ ]:
import subprocess
import sys

demos = [
    ("anomaly_detector.py", "autoencoder", ["--epochs", "2", "--image-size", "64", "--device", "cpu"]),
    ("pixel_reference_detector.py", "pixel_reference", ["--image-size", "64"]),
    ("patch_knn_detector.py", "patch_knn", ["--image-size", "64"]),
]
for script, name, options in demos:
    output_dir = project_root / "artifacts" / f"colab_{name}"
    command = [sys.executable, script, "demo", "--output-dir", str(output_dir), *options]
    print(f"\n執行 {script} ...")
    subprocess.run(command, cwd=project_root, check=True)

In [ ]:
from IPython.display import display
from PIL import Image

for _, name, _ in demos:
    result_dir = project_root / "artifacts" / f"colab_{name}" / "demo_results"
    print(f"\n{name}")
    for suffix in ("clean", "defect"):
        overlay = result_dir / f"demo_{suffix}_overlay.png"
        if overlay.is_file():
            print(f"{suffix} overlay")
            display(Image.open(overlay))
        result_json = result_dir / f"demo_{suffix}_result.json"
        if result_json.is_file():
            print(result_json.read_text(encoding="utf-8"))